# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Distributions of the key fields, before trusting any comparison built on their means.

In [1]:
import pandas as pd
import os

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

for col in ["ctr", "engagement_rate", "impressions_90d", "clicks_90d"]:
    desc = df[col].describe(percentiles=[.5, .9, .99])
    print(f"--- {col} ---")
    print(desc.round(3))
    print(f"  mean/median ratio: {desc['mean'] / max(desc['50%'], 1e-9):.1f}x  <- heavy tail if >> 1\n")

--- ctr ---
count    30000.000
mean         0.511
std          3.279
min          0.000
50%          0.070
90%          0.650
99%          8.330
max        100.000
Name: ctr, dtype: float64
  mean/median ratio: 7.3x  <- heavy tail if >> 1

--- engagement_rate ---
count    30000.000
mean         2.535
std          8.310
min          0.000
50%          0.000
90%          6.940
99%         33.330
max        100.000
Name: engagement_rate, dtype: float64
  mean/median ratio: 2534520000.0x  <- heavy tail if >> 1

--- impressions_90d ---
count     30000.000
mean       5200.366
std       16838.020
min           1.000
50%         731.000
90%       12136.400
99%       73505.830
max      517715.000
Name: impressions_90d, dtype: float64
  mean/median ratio: 7.1x  <- heavy tail if >> 1

--- clicks_90d ---
count    30000.000
mean        16.097
std         75.077
min          0.000
50%          1.000
90%         32.000
99%        253.010
max       4178.000
Name: clicks_90d, dtype: float64
  mean/medi

## 2. Signal test #1 / #2 / #3 (verdict each)

Three signals a content manager would naturally reach for, each checked at the **mean** (what a
quick groupby shows) and the **median** (what actually happens to a typical page) -- because the
heavy tails above mean a handful of outlier pages can make a mean say something the typical page
doesn't back up.

In [2]:
for col, story in [
    ("ctr", "lower CTR -> more likely declining"),
    ("engagement_rate", "lower engagement -> more likely declining"),
    ("days_since_last_update", "longer since update -> more likely declining"),
]:
    g = df.groupby("is_declining_label")[col].agg(["mean", "median"]).round(3)
    mean_direction = "CONFIRMED" if g.loc[1, "mean"] < g.loc[0, "mean"] else "OPPOSITE"
    median_direction = "CONFIRMED" if g.loc[1, "median"] < g.loc[0, "median"] else (
        "FLAT" if g.loc[1, "median"] == g.loc[0, "median"] else "OPPOSITE")
    verdict = "CONFIRMED" if mean_direction == median_direction == "CONFIRMED" else "MIXED"
    print(f"--- {col} ({story}) ---")
    print(g)
    print(f"  mean says: {mean_direction}  |  median says: {median_direction}  ->  VERDICT: {verdict}\n")

print("Takeaway: CTR's mean says CONFIRMED, but its MEDIAN FLIPS -- the typical declining page")
print("actually has a slightly HIGHER CTR than the typical stable page; a few very-high-CTR")
print("stable pages are pulling that mean up. Engagement rate is flat at the median for both")
print("groups (>50% of all pages show zero measured engagement). None of these three signals")
print("survives as a clean, median-level separator alone -- which is exactly why Lane 2 uses a")
print("model that weighs many signals together (w05_model.ipynb) rather than a single-field rule.")

--- ctr (lower CTR -> more likely declining) ---
                     mean  median
is_declining_label               
0                   0.732    0.04
1                   0.324    0.08
  mean says: CONFIRMED  |  median says: OPPOSITE  ->  VERDICT: MIXED

--- engagement_rate (lower engagement -> more likely declining) ---
                     mean  median
is_declining_label               
0                   2.650     0.0
1                   2.437     0.0
  mean says: CONFIRMED  |  median says: FLAT  ->  VERDICT: MIXED

--- days_since_last_update (longer since update -> more likely declining) ---
                      mean  median
is_declining_label                
0                   42.373    20.0
1                   49.246    20.0
  mean says: OPPOSITE  |  median says: FLAT  ->  VERDICT: MIXED

Takeaway: CTR's mean says CONFIRMED, but its MEDIAN FLIPS -- the typical declining page
actually has a slightly HIGHER CTR than the typical stable page; a few very-high-CTR
stable pages are pu

## 3. The flag-linked test

FlyRank's baseline rule (`w04_baseline_score.ipynb`) flags a page if it's **91-180 days stale**
AND still **visible** (`impressions_90d >= 300`). Does staleness actually track with higher
decline rates in this data, or is that assumption unsupported?

In [3]:
print("Decline rate by freshness tier:")
print(df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).round(3))

stale = (df["freshness_tier"] == "91-180")
visible = (df["impressions_90d"] >= 300)
flagged = stale & visible
print("\nDecline rate: baseline-flagged pages vs. everyone else:")
print(df.groupby(flagged)["is_declining_label"].agg(["mean", "count"]).round(3))

Decline rate by freshness tier:
                 mean  count
freshness_tier              
0-30            0.511  20480
181+            0.471    174
31-90           0.589    175
91-180          0.611   9171

Decline rate: baseline-flagged pages vs. everyone else:
        mean  count
False  0.518  22788
True   0.617   7212


## 4. What this means in practice

The `91-180` freshness tier does show the highest decline rate of the four tiers (61.1% vs. 51.1%
for fresh 0-30 pages), and the full baseline rule (stale AND visible) flags a group with a 61.7%
decline rate against 51.8% for everyone else -- the rule's core assumption is **directionally
CONFIRMED**, but the lift is modest (about 10 points), not the dramatic separator its "one clean
rule" framing might suggest.

For a content team: treat any single-field rule of thumb -- staleness, CTR, engagement -- as a
weak, directionally-useful hint, never a standalone trigger. The heavy tails mean a quick
mean-based dashboard comparison can overstate how separable declining pages really are; that gap
is exactly why the ranking model in `w05_model.ipynb` earns its place over a simpler rule.

In [4]:
# Receipt for the 10-point-lift claim above
lift = df.groupby(flagged)["is_declining_label"].mean()
print(f"Decline rate, baseline-flagged pages: {lift[True]:.1%}")
print(f"Decline rate, everyone else:          {lift[False]:.1%}")
print(f"Lift: {(lift[True] - lift[False]) * 100:.1f} percentage points")

Decline rate, baseline-flagged pages: 61.7%
Decline rate, everyone else:          51.8%
Lift: 9.9 percentage points


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms already shipped in the dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.